## Separação Temporal entre Treino e Validação

### Objetivo

Separar os dados de treinamento e validação respeitando a ordem temporal das vendas, simulando o cenário real em que o modelo utiliza o histórico disponível para prever períodos futuros.

### Hipótese

Uma validação baseada em períodos futuros, sem embaralhamento das observações, deve fornecer uma avaliação mais realista da capacidade do modelo de generalizar para novas datas.


### Estratégia
O período final de 28 dias será reservado para validação, enquanto as observações anteriores serão utilizadas no treinamento.

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# CARREGAMENTO DA BASE DE FEATURES
# ============================================================

sales_path = "../data/processed/sales_features.parquet"

sales_long = pd.read_parquet(sales_path)

print("Base carregada com sucesso.")
print(f"Dimensões: {sales_long.shape}")

display(sales_long.head())

Base carregada com sucesso.
Dimensões: (5832737, 27)


,item_id,store_id,d,sales,lag_1,lag_7,lag_14,lag_28,date,wday,...,snap,event_name_1,event_type_1,event_name_2,event_type_2,day_num,_day_num,rolling_mean_7,rolling_mean_14,rolling_mean_28
0,FOODS_1_001,CA_1,d_1,3,NaN,NaN,NaN,NaN,2011-01-29,1,...,0,NaN,NaN,NaN,NaN,1,1,NaN,NaN,NaN
1,FOODS_1_001,CA_1,d_2,0,3.0,NaN,NaN,NaN,2011-01-30,2,...,0,NaN,NaN,NaN,NaN,2,2,NaN,NaN,NaN
2,FOODS_1_001,CA_1,d_3,0,0.0,NaN,NaN,NaN,2011-01-31,3,...,0,NaN,NaN,NaN,NaN,3,3,NaN,NaN,NaN
3,FOODS_1_001,CA_1,d_4,1,0.0,NaN,NaN,NaN,2011-02-01,4,...,1,NaN,NaN,NaN,NaN,4,4,NaN,NaN,NaN
4,FOODS_1_001,CA_1,d_5,4,1.0,NaN,NaN,NaN,2011-02-02,5,...,1,NaN,NaN,NaN,NaN,5,5,NaN,NaN,NaN


In [17]:
# ============================================================
# SEPARAÇÃO TEMPORAL ENTRE TREINO E VALIDAÇÃO
# ============================================================

# Garante que a coluna de data esteja no formato datetime
sales_long["date"] = pd.to_datetime(sales_long["date"])

# Define o último dia disponível na base
last_date = sales_long["date"].max()

# Reserva os últimos 28 dias para validação
validation_start = last_date - pd.Timedelta(days=27)

# Define o primeiro dia em que as features históricas estão completas
first_model_date = (
    sales_long["date"].min()
    + pd.Timedelta(days=28)
)

# Separa o período de aquecimento
# e mantém no treino apenas observações com histórico suficiente
train_data = sales_long[
    (sales_long["date"] >= first_model_date) &
    (sales_long["date"] < validation_start)
].copy()

# Separa os últimos 28 dias para validação
validation_data = sales_long[
    sales_long["date"] >= validation_start
].copy()

print("Período de aquecimento:")
print(
    f"{sales_long['date'].min()} até "
    f"{first_model_date - pd.Timedelta(days=1)}"
)

print("\nPeríodo de treino:")
print(
    f"{train_data['date'].min()} até "
    f"{train_data['date'].max()}"
)

print("\nPeríodo de validação:")
print(
    f"{validation_data['date'].min()} até "
    f"{validation_data['date'].max()}"
)

print("\nDimensões:")
print(f"Treino: {train_data.shape}")
print(f"Validação: {validation_data.shape}")

Período de aquecimento:
2011-01-29 00:00:00 até 2011-02-25 00:00:00

Período de treino:
2011-02-26 00:00:00 até 2016-03-27 00:00:00

Período de validação:
2016-03-28 00:00:00 até 2016-04-24 00:00:00

Dimensões:
Treino: (5661993, 27)
Validação: (85372, 27)


In [18]:
# ============================================================
# VALIDAÇÃO DA SEPARAÇÃO TEMPORAL
# ============================================================

print("Última data do treino:")
print(train_data["date"].max())

print("\nPrimeira data da validação:")
print(validation_data["date"].min())

print("\nÚltima data da validação:")
print(validation_data["date"].max())

print("\nQuantidade de observações:")
print(f"Treino: {len(train_data):,}")
print(f"Validação: {len(validation_data):,}")

print("\nSobreposição temporal:")
print(
    train_data["date"].max() >= validation_data["date"].min()
)

Última data do treino:
2016-03-27 00:00:00

Primeira data da validação:
2016-03-28 00:00:00

Última data da validação:
2016-04-24 00:00:00

Quantidade de observações:
Treino: 5,661,993
Validação: 85,372

Sobreposição temporal:
False


### Resultado

A base foi separada cronologicamente, preservando a ordem temporal e considerando o período de aquecimento necessário para o cálculo das variáveis históricas.

- Período de aquecimento: 2011-01-29 até 2011-02-25
- Período de treino: 2011-02-26 até 2016-03-27
- Período de validação: 2016-03-28 até 2016-04-24
- Observações de treino: 5.661.993
- Observações de validação: 85.372
- Sobreposição temporal: não identificada

### Decisão

Os últimos 28 dias foram reservados como conjunto de validação, mantendo a separação temporal para simular o cenário real de previsão de períodos futuros.

## Preparação dos Dados para Modelagem

### Objetivo

Preparar os conjuntos de treino e validação para receber os modelos de previsão, definindo a variável alvo, as variáveis preditoras e o tratamento necessário para valores ausentes e variáveis categóricas.

A preparação será realizada de forma a preservar a separação temporal e evitar o uso de informações do período de validação durante o treinamento.

In [19]:
# ============================================================
# DEFINIÇÃO DA VARIÁVEL ALVO E DAS FEATURES
# ============================================================

target = "sales"

# Colunas que não serão utilizadas diretamente como preditoras
exclude_cols = [
    "sales",
    "date"
]

feature_cols = [
    col for col in sales_long.columns
    if col not in exclude_cols
]

print("Variável alvo:")
print(target)

print("\nQuantidade de features:")
print(len(feature_cols))

print("\nFeatures:")
print(feature_cols)

Variável alvo:
sales

Quantidade de features:
25

Features:
['item_id', 'store_id', 'd', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']


In [20]:
# ============================================================
# SEPARAÇÃO ENTRE FEATURES E VARIÁVEL ALVO
# ============================================================

X_train = train_data[feature_cols].copy()
y_train = train_data[target].copy()

X_validation = validation_data[feature_cols].copy()
y_validation = validation_data[target].copy()

print("Dimensões de X_train:")
print(X_train.shape)

print("\nDimensões de y_train:")
print(y_train.shape)

print("\nDimensões de X_validation:")
print(X_validation.shape)

print("\nDimensões de y_validation:")
print(y_validation.shape)

Dimensões de X_train:
(5661993, 25)

Dimensões de y_train:
(5661993,)

Dimensões de X_validation:
(85372, 25)

Dimensões de y_validation:
(85372,)


## Preparação dos Dados para Modelagem

### Objetivo

Estruturar os conjuntos de treino e validação nas variáveis de entrada (`X`) e na variável alvo (`y`), mantendo a mesma estrutura de features entre os períodos.

A variável `sales` será utilizada como alvo da previsão, enquanto as demais variáveis selecionadas serão utilizadas como informações explicativas.

In [21]:
# ============================================================
# PREPARAÇÃO DOS DADOS PARA MODELAGEM
# ============================================================

target = "sales"

# Variáveis que não serão utilizadas diretamente como features
exclude_cols = ["sales", "date"]

# Features
feature_cols = [
    col for col in train_data.columns
    if col not in exclude_cols
]

# Conjunto de treino
X_train = train_data[feature_cols].copy()
y_train = train_data[target].copy()

# Conjunto de validação
X_validation = validation_data[feature_cols].copy()
y_validation = validation_data[target].copy()

# Conferências
print("Dimensões:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_validation: {X_validation.shape}")
print(f"y_validation: {y_validation.shape}")

print("\nVariável alvo:")
print(target)

print("\nQuantidade de features:")
print(len(feature_cols))

print("\nFeatures:")
print(feature_cols)

Dimensões:
X_train: (5661993, 25)
y_train: (5661993,)
X_validation: (85372, 25)
y_validation: (85372,)

Variável alvo:
sales

Quantidade de features:
25

Features:
['item_id', 'store_id', 'd', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']


In [22]:
# ============================================================
# DIAGNÓSTICO DAS FEATURES
# ============================================================

print("Tipos das features:")
print(X_train.dtypes)

print("\nValores ausentes:")
print(
    X_train.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nFeatures categóricas:")
print(
    X_train.select_dtypes(
        include=["category", "str"]
    ).columns.tolist()
)

print("\nFeatures numéricas:")
print(
    X_train.select_dtypes(
        include=["number"]
    ).columns.tolist()
)

Tipos das features:
item_id            category
store_id           category
d                       str
lag_1               float32
lag_7               float32
lag_14              float32
lag_28              float32
wday                   int8
weekday            category
month                  int8
year                  int16
week_of_year           int8
snap_CA                int8
snap_TX                int8
snap_WI                int8
snap                   int8
event_name_1       category
event_type_1       category
event_name_2       category
event_type_2       category
day_num               int16
_day_num              int32
rolling_mean_7      float32
rolling_mean_14     float32
rolling_mean_28     float32
dtype: object

Valores ausentes:
event_type_2       5649797
event_name_2       5649797
event_name_1       5201594
event_type_1       5201594
lag_7                    0
store_id                 0
d                        0
lag_1                    0
item_id                  0
week

In [23]:
# ============================================================
# VERIFICAÇÃO DE VALORES AUSENTES
# ============================================================

print("Valores ausentes no treino:")
print(
    X_train.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nValores ausentes na validação:")
print(
    X_validation.isna()
    .sum()
    .sort_values(ascending=False)
)

Valores ausentes no treino:
event_type_2       5649797
event_name_2       5649797
event_name_1       5201594
event_type_1       5201594
lag_7                    0
store_id                 0
d                        0
lag_1                    0
item_id                  0
weekday                  0
wday                     0
lag_28                   0
lag_14                   0
snap_CA                  0
month                    0
week_of_year             0
year                     0
snap                     0
snap_WI                  0
snap_TX                  0
day_num                  0
_day_num                 0
rolling_mean_7           0
rolling_mean_14          0
rolling_mean_28          0
dtype: int64

Valores ausentes na validação:
event_name_1       85372
event_type_1       85372
event_name_2       85372
event_type_2       85372
lag_7                  0
store_id               0
d                      0
lag_1                  0
item_id                0
weekday                0
wd

In [24]:
# ============================================================
# TRATAMENTO DAS VARIÁVEIS CATEGÓRICAS DE EVENTOS
# ============================================================

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:
    X_train[col] = X_train[col].astype("category")
    X_validation[col] = X_validation[col].astype("category")

    X_train[col] = X_train[col].cat.add_categories(["NoEvent"]).fillna("NoEvent")
    X_validation[col] = X_validation[col].cat.add_categories(["NoEvent"]).fillna("NoEvent")

print("Valores ausentes nas variáveis de eventos - treino:")
print(X_train[event_cols].isna().sum())

print("\nValores ausentes nas variáveis de eventos - validação:")
print(X_validation[event_cols].isna().sum())

Valores ausentes nas variáveis de eventos - treino:
event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64

Valores ausentes nas variáveis de eventos - validação:
event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64


In [25]:
# ============================================================
# CHECAGEM FINAL DAS FEATURES PARA MODELAGEM
# ============================================================

print("Quantidade de features:")
print(f"Treino: {X_train.shape[1]}")
print(f"Validação: {X_validation.shape[1]}")

print("\nFeatures iguais entre treino e validação:")
print(X_train.columns.tolist() == X_validation.columns.tolist())

print("\nValores ausentes no treino:")
print(X_train.isna().sum().sum())

print("\nValores ausentes na validação:")
print(X_validation.isna().sum().sum())

print("\nVariável alvo está fora das features:")
print("sales" not in X_train.columns)

print("\nLista de features:")
print(X_train.columns.tolist())

Quantidade de features:
Treino: 25
Validação: 25

Features iguais entre treino e validação:
True

Valores ausentes no treino:
0

Valores ausentes na validação:
0

Variável alvo está fora das features:
True

Lista de features:
['item_id', 'store_id', 'd', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'wday', 'weekday', 'month', 'year', 'week_of_year', 'snap_CA', 'snap_TX', 'snap_WI', 'snap', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'day_num', '_day_num', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']


In [26]:
# ============================================================
# CHECAGEM FINAL DE VALORES AUSENTES
# ============================================================

print("Valores ausentes no treino:")
print(X_train.isna().sum().sum())

print("\nValores ausentes na validação:")
print(X_validation.isna().sum().sum())

Valores ausentes no treino:
0

Valores ausentes na validação:
0


In [29]:
# ============================================================
# EXPORTAÇÃO DOS DADOS PREPARADOS PARA MODELAGEM
# ============================================================

train_modeling = X_train.copy()
train_modeling[target] = y_train.values

validation_modeling = X_validation.copy()
validation_modeling[target] = y_validation.values

train_modeling.to_parquet(
    "../data/processed/train_modeling.parquet",
    index=False
)

validation_modeling.to_parquet(
    "../data/processed/validation_modeling.parquet",
    index=False
)

print("Dados preparados para modelagem salvos com sucesso.")

print(f"Treino: {train_modeling.shape}")
print(f"Validação: {validation_modeling.shape}")

print("\nArquivos:")
print("../data/processed/train_modeling.parquet")
print("../data/processed/validation_modeling.parquet")

Dados preparados para modelagem salvos com sucesso.
Treino: (5661993, 26)
Validação: (85372, 26)

Arquivos:
../data/processed/train_modeling.parquet
../data/processed/validation_modeling.parquet


Fechamento da Preparação dos Dados

Os conjuntos de treino e validação foram preparados mantendo a separação temporal definida anteriormente.

Foram definidas a variável alvo (sales) e as variáveis preditoras, mantendo a mesma estrutura de features entre treino e validação.

Após o tratamento dos dados, os conjuntos finais foram exportados para utilização na etapa de modelagem.

A etapa seguinte consiste na avaliação e comparação dos modelos de previsão.